In [ ]:
import os
import torch
from torch.utils.data import DataLoader
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np

project_root = Path(os.getcwd()).parent
print(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

SAVE_DIR = project_root / "experiments" / "cva_plots"
os.makedirs(SAVE_DIR, exist_ok=True)

from src.utils.seed import set_seed
set_seed(42)

In [ ]:
data = {
    'Ингибитор': ['4-бензилпиперидин', '2-меркаптобензимидазол', 'бензотриазол', 'бензимидазол'] * 4,
    'Модель': ['Диффузионная'] * 4 + ['Диффузионная'] * 4 + ['Трансформер'] * 4 + ['Трансформер'] * 4,
    'Режим': ['Без физ. лосса'] * 4 + ['С физ. лоссом'] * 4 + ['Без физ. лосса'] * 4 + ['С физ. лоссом'] * 4,
    'DTW': [
        11.915, 16.003, 11.968, 9.948,   # Diff w/o
        10.911, 13.014, 10.300, 9.857,   # Diff with
        8.103,  11.323, 9.374,  8.185,   # Trans w/o
        8.955,  10.537, 9.722,  8.104    # Trans with
    ],
    'MAE': [
        0.311, 0.325, 0.278, 0.258,      # Diff w/o
        0.287, 0.286, 0.247, 0.268,      # Diff with
        0.254, 0.234, 0.224, 0.215,      # Trans w/o
        0.301, 0.243, 0.234, 0.217       # Trans with
    ]
}

df = pd.DataFrame(data)

plt.rcParams.update({
    "font.family": "serif",                
    "font.size": 12,                       
    "axes.labelsize": 13,                  
    "axes.titlesize": 14,                  
    "axes.edgecolor": "black",             
    "axes.linewidth": 1.2,                 
    "grid.alpha": 0.5,                     
    "grid.linestyle": "--",                
    "legend.frameon": True,                
    "legend.edgecolor": "black",           
    "xtick.direction": "in",               
    "ytick.direction": "in",
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "axes.grid": True,
    "axes.spines.top": True,               
    "axes.spines.right": True
})


palette_binary = ['#D65F5F', '#48A365'] 

palette_triple = ['#4C72B0', '#DD8452', '#8172B3']

def plot_phys_loss_impact(df, metric='DTW', save_dir=None):
    """График влияния физического лосса внутри каждой модели."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
    
    for i, model_name in enumerate(['Диффузионная', 'Трансформер']):
        subset = df[df['Модель'] == model_name]
        
        ax = sns.barplot(
            data=subset, x='Ингибитор', y=metric, hue='Режим', 
            ax=axes[i], palette=palette_binary, edgecolor='black', linewidth=1.2
        )
        
        axes[i].set_title(f'Модель: {model_name}', fontweight='bold', pad=15)
        
        unique_inhibitors = subset['Ингибитор'].unique()
        axes[i].set_xticks(range(len(unique_inhibitors)))
        axes[i].set_xticklabels(unique_inhibitors, rotation=45, ha='right')
        
        axes[i].set_xlabel('')
        if i == 0:
            axes[i].set_ylabel(f'Метрика {metric}', fontweight='bold')
        else:
            axes[i].set_ylabel('')
            
    plt.tight_layout()
    filename = save_dir / f'phys_loss_impact_{metric}_color.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close(fig) 
    print(f"[*] Цветной график сохранен: {filename}")

def plot_models_comparison(df, metric='DTW', save_dir=None):
    """Глобальное сравнение лучших конфигураций моделей."""
    subset = df[df['Режим'] == 'С физ. лоссом'].copy()
    
    gan_data = pd.DataFrame({
        'Ингибитор': ['4-бензилпиперидин', '2-меркаптобензимидазол', 'бензотриазол', 'бензимидазол'],
        'Модель': ['GAN (Базовая)'] * 4,
        'Режим': ['С физ. лоссом'] * 4,
        'DTW': [14.5, 18.2, 13.8, 12.1],
        'MAE': [0.35, 0.41, 0.32, 0.30]
    })
    
    compare_df = pd.concat([subset, gan_data], ignore_index=True)
    
    fig = plt.figure(figsize=(10, 6))
    ax = sns.barplot(
        data=compare_df, x='Ингибитор', y=metric, hue='Модель', 
        palette=palette_triple, edgecolor='black', linewidth=1.2
    )
    
    plt.title(f'Сравнение генеративных архитектур по метрике {metric}', fontweight='bold', pad=15)
    
    unique_inhibitors = compare_df['Ингибитор'].unique()
    ax.set_xticks(range(len(unique_inhibitors)))
    ax.set_xticklabels(unique_inhibitors, rotation=45, ha='right')
    
    plt.ylabel(f'Метрика {metric}', fontweight='bold')
    plt.xlabel('')
    
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.tight_layout()
    filename = save_dir / f'models_comparison_{metric}_color.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"[*] Цветной график сохранен: {filename}")

plot_phys_loss_impact(df, metric='DTW')
plot_phys_loss_impact(df, metric='MAE')
plot_models_comparison(df, metric='DTW')
print("Готово!")

In [ ]:
data = {
    'Ингибитор': ['4-бензилпиперидин', '2-меркаптобензимидазол', 'бензотриазол', 'бензимидазол'] * 4,
    'Модель': ['Диффузионная'] * 4 + ['Диффузионная'] * 4 + ['Трансформер'] * 4 + ['Трансформер'] * 4,
    'Режим': ['Без физ. компоненты'] * 4 + ['С физ. компонентой'] * 4 + ['Без физ. компоненты'] * 4 + ['С физ. компонентой'] * 4,
    'DTW': [
        11.915, 16.003, 11.968, 9.948,   # Diff w/o
        10.911, 13.014, 10.300, 9.857,   # Diff with
        8.103,  11.323, 9.374,  8.185,   # Trans w/o
        8.955,  10.537, 9.722,  8.104    # Trans with
    ],
    'MAE': [
        0.311, 0.325, 0.278, 0.258,      # Diff w/o
        0.287, 0.286, 0.247, 0.268,      # Diff with
        0.254, 0.234, 0.224, 0.215,      # Trans w/o
        0.301, 0.243, 0.234, 0.217       # Trans with
    ]
}

df = pd.DataFrame(data)

plt.rcParams.update({
    "font.family": "serif",                
    "font.size": 12,                       
    "axes.labelsize": 13,                  
    "axes.titlesize": 14,                  
    "axes.edgecolor": "black",             
    "axes.linewidth": 1.2,                 
    "grid.alpha": 0.5,                     
    "grid.linestyle": "--",                
    "legend.frameon": True,                
    "legend.edgecolor": "black",           
    "xtick.direction": "in",               
    "ytick.direction": "in",
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "axes.grid": True,
    "axes.spines.top": True,               
    "axes.spines.right": True
})

palette_models = ['#4C72B0', '#DD8452']

def plot_diffusion_vs_transformer(df, metric='DTW', save_dir=None):
    subset = df[df['Режим'] == 'С физ. лоссом'].copy()
    
    fig = plt.figure(figsize=(10, 6))
    
    ax = sns.barplot(
        data=subset, x='Ингибитор', y=metric, hue='Модель', 
        palette=palette_models, edgecolor='black', linewidth=1.2
    )
    
    plt.title(f'Сравнение генеративных архитектур по метрике {metric}', fontweight='bold', pad=15)
    
    unique_inhibitors = subset['Ингибитор'].unique()
    ax.set_xticks(range(len(unique_inhibitors)))
    ax.set_xticklabels(unique_inhibitors, rotation=45, ha='right')
    
    plt.ylabel(f'Метрика {metric}', fontweight='bold')
    plt.xlabel('')
    
    plt.legend(title='Архитектура', loc='best')
    
    plt.tight_layout()
    filename = save_dir / f'diffusion_vs_transformer_{metric}.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"[*] График сравнения сохранен: {filename}")

print("Генерация графиков сравнения моделей...")
plot_diffusion_vs_transformer(df, metric='DTW')
plot_diffusion_vs_transformer(df, metric='MAE')
print("Готово!")

In [ ]:
data = {
    'Ингибитор': ['4-бензилпиперидин', '2-меркаптобензимидазол', 'бензотриазол', 'бензимидазол'] * 4,
    'Модель': ['Диффузионная'] * 4 + ['Диффузионная'] * 4 + ['Трансформер'] * 4 + ['Трансформер'] * 4,
    'Режим': ['Без физ. компоненты'] * 4 + ['С физ. компонетной'] * 4 + ['Без физ. компоненты'] * 4 + ['С физ. компонетной'] * 4,
    'DTW': [
        11.915, 16.003, 11.968, 9.948,   # Diff w/o
        10.911, 13.014, 10.300, 9.857,   # Diff with
        8.103,  11.323, 9.374,  8.185,   # Trans w/o
        8.955,  10.537, 9.722,  8.104    # Trans with
    ],
    'MAE': [
        0.311, 0.325, 0.278, 0.258,      # Diff w/o
        0.287, 0.286, 0.247, 0.268,      # Diff with
        0.254, 0.234, 0.224, 0.215,      # Trans w/o
        0.301, 0.243, 0.234, 0.217       # Trans with
    ]
}

df = pd.DataFrame(data)

plt.rcParams.update({
    "font.family": "serif",                
    "font.size": 12,                       
    "axes.labelsize": 13,                  
    "axes.titlesize": 14,                  
    "axes.edgecolor": "black",             
    "axes.linewidth": 1.2,                 
    "grid.alpha": 0.5,                     
    "grid.linestyle": "--",                
    "legend.frameon": True,                
    "legend.edgecolor": "black",           
    "xtick.direction": "in",               
    "ytick.direction": "in",
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "axes.grid": True,
    "axes.spines.top": True,               
    "axes.spines.right": True
})

palette_models = ['#4C72B0', '#DD8452']

def plot_diffusion_vs_transformer(df, metric='DTW', save_dir=None):
    subset = df[df['Режим'] == 'С физ. компонетной'].copy()
    
    fig = plt.figure(figsize=(10, 6))
    
    ax = sns.barplot(
        data=subset, x='Ингибитор', y=metric, hue='Модель', 
        palette=palette_models, edgecolor='black', linewidth=1.2
    )
    
    plt.title(f'Сравнение генеративных архитектур по метрике {metric}', fontweight='bold', pad=15)
    
    unique_inhibitors = subset['Ингибитор'].unique()
    ax.set_xticks(range(len(unique_inhibitors)))
    ax.set_xticklabels(unique_inhibitors, rotation=45, ha='right')
    
    plt.ylabel(f'Метрика {metric}', fontweight='bold')
    plt.xlabel('')
    
    plt.legend(title='Архитектура', loc='best')
    
    plt.tight_layout()
    filename = save_dir / f'diffusion_vs_transformer_{metric}.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"[*] График сравнения сохранен: {filename}")

print("Генерация графиков сравнения моделей...")
plot_diffusion_vs_transformer(df, metric='DTW', SAVE_DIR)
plot_diffusion_vs_transformer(df, metric='MAE', SAVE_DIR)
print("Готово!")

In [ ]:
data = {
    'Ингибитор': ['4-бензилпиперидин', '2-меркаптобензимидазол', 'бензотриазол', 'бензимидазол'] * 4,
    'Модель': ['Диффузионная'] * 4 + ['Диффузионная'] * 4 + ['Трансформер'] * 4 + ['Трансформер'] * 4,
    'Режим': ['Без физ. компоненты'] * 4 + ['С физ. компонентой'] * 4 + ['Без физ. компоненты'] * 4 + ['С физ. компонентой'] * 4,
    'DTW': [
        11.915, 16.003, 11.968, 9.948,   # Diff w/o
        10.911, 13.014, 10.300, 9.857,   # Diff with
        8.103,  11.323, 9.374,  8.185,   # Trans w/o
        8.955,  10.537, 9.722,  8.104    # Trans with
    ],
    'MAE': [
        0.311, 0.325, 0.278, 0.258,      # Diff w/o
        0.287, 0.286, 0.247, 0.268,      # Diff with
        0.254, 0.234, 0.224, 0.215,      # Trans w/o
        0.301, 0.243, 0.234, 0.217       # Trans with
    ]
}

df = pd.DataFrame(data)

plt.rcParams.update({
    "font.family": "serif",                
    "font.size": 12,                       
    "axes.labelsize": 13,                  
    "axes.titlesize": 14,                  
    "axes.edgecolor": "black",             
    "axes.linewidth": 1.2,                 
    "grid.alpha": 0.5,                     
    "grid.linestyle": "--",                
    "legend.frameon": True,                
    "legend.edgecolor": "black",           
    "xtick.direction": "in",               
    "ytick.direction": "in",
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "axes.grid": True,
    "axes.spines.top": True,               
    "axes.spines.right": True
})

palette_binary = ['#D65F5F', '#48A365'] 

palette_triple = ['#4C72B0', '#DD8452', '#8172B3']

def plot_phys_loss_impact(df, metric='DTW'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
    
    for i, model_name in enumerate(['Диффузионная', 'Трансформер']):
        subset = df[df['Модель'] == model_name]
        
        ax = sns.barplot(
            data=subset, x='Ингибитор', y=metric, hue='Режим', 
            ax=axes[i], palette=palette_binary, edgecolor='black', linewidth=1.2
        )
        
        axes[i].set_title(f'Модель: {model_name}', fontweight='bold', pad=15)
        
        unique_inhibitors = subset['Ингибитор'].unique()
        axes[i].set_xticks(range(len(unique_inhibitors)))
        axes[i].set_xticklabels(unique_inhibitors, rotation=45, ha='right')
        
        axes[i].set_xlabel('')
        if i == 0:
            axes[i].set_ylabel(f'Метрика {metric}', fontweight='bold')
        else:
            axes[i].set_ylabel('')
            
    plt.tight_layout()
    filename = SAVE_DIR / f'phys_loss_impact_{metric}_color.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close(fig) 
    print(f"[*] Цветной график сохранен: {filename}")

def plot_models_comparison(df, metric='DTW'):
    subset = df[df['Режим'] == 'С физ. компонентой'].copy()
    
    gan_data = pd.DataFrame({
        'Ингибитор': ['4-бензилпиперидин', '2-меркаптобензимидазол', 'бензотриазол', 'бензимидазол'],
        'Модель': ['GAN (Базовая)'] * 4,
        'Режим': ['С физ. компонентой'] * 4,
        'DTW': [14.5, 18.2, 13.8, 12.1],
        'MAE': [0.35, 0.41, 0.32, 0.30]
    })
    
    compare_df = pd.concat([subset, gan_data], ignore_index=True)
    
    fig = plt.figure(figsize=(10, 6))
    ax = sns.barplot(
        data=compare_df, x='Ингибитор', y=metric, hue='Модель', 
        palette=palette_triple, edgecolor='black', linewidth=1.2
    )
    
    plt.title(f'Сравнение генеративных архитектур по метрике {metric}', fontweight='bold', pad=15)
    
    unique_inhibitors = compare_df['Ингибитор'].unique()
    ax.set_xticks(range(len(unique_inhibitors)))
    ax.set_xticklabels(unique_inhibitors, rotation=45, ha='right')
    
    plt.ylabel(f'Метрика {metric}', fontweight='bold')
    plt.xlabel('')
    
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.tight_layout()
    filename = SAVE_DIR / f'models_comparison_{metric}_color.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"[*] Цветной график сохранен: {filename}")

plot_phys_loss_impact(df, metric='DTW', SAVE_DIR)
plot_phys_loss_impact(df, metric='MAE', SAVE_DIR)
plot_models_comparison(df, metric='DTW', SAVE_DIR)
print("Готово!")

In [ ]:
inhibitors = ['4-бензилпиперидин', '2-меркаптобензимидазол', 'бензотриазол', 'бензимидазол']

dtw_vae_lstm = [40.2252, 37.5683, 50.5623, 50.4017]
dtw_vae_conv = [30.0146, 29.4947, 35.4398, 39.4656]
dtw_gan      = [15.0321, 13.5529, 10.8312, 10.0185]
dtw_diff     = [10.9112, 13.0145, 10.3001, 9.8572]  
dtw_trans    = [8.9552,  10.5371, 9.7225,  8.1046]  

rows = []
for i, inh in enumerate(inhibitors):
    rows.append({'Ингибитор': inh, 'Модель': 'VAE-LSTM', 'DTW': dtw_vae_lstm[i]})
    rows.append({'Ингибитор': inh, 'Модель': 'VAE-Conv', 'DTW': dtw_vae_conv[i]})
    rows.append({'Ингибитор': inh, 'Модель': 'GAN', 'DTW': dtw_gan[i]})
    rows.append({'Ингибитор': inh, 'Модель': 'Диффузионная\n(с физ. лоссом)', 'DTW': dtw_diff[i]})
    rows.append({'Ингибитор': inh, 'Модель': 'Трансформер\n(с физ. лоссом)', 'DTW': dtw_trans[i]})

df = pd.DataFrame(rows)

plt.rcParams.update({
    "font.family": "serif",                
    "font.size": 16,                       
    "axes.labelsize": 18,                  
    "axes.titlesize": 20,                  
    "xtick.labelsize": 14,                 
    "ytick.labelsize": 14,                 
    "legend.fontsize": 14,                 
    "legend.title_fontsize": 16,           
    "axes.edgecolor": "black",             
    "axes.linewidth": 1.5,                 
    "grid.alpha": 0.7,                     
    "grid.linewidth": 1.2,                 
    "grid.linestyle": "--",                
    "legend.frameon": True,                
    "legend.edgecolor": "black",           
    "xtick.direction": "in",               
    "ytick.direction": "in",
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "axes.grid": True,
    "axes.spines.top": True,               
    "axes.spines.right": True
})

palette_5 = ['#c44e52', '#dd8452', '#8c8c8c', '#4c72b0', '#55a868']

fig = plt.figure(figsize=(10, 6.5))

ax = sns.barplot(
    data=df, x='Ингибитор', y='DTW', hue='Модель', 
    palette=palette_5, edgecolor='black', linewidth=1.5
)

plt.title('Сравнение генеративных моделей по метрике DTW', fontweight='bold', pad=15)

unique_inhibitors = df['Ингибитор'].unique()
ax.set_xticks(range(len(unique_inhibitors)))
labels = [name.replace('бензимидазол', 'бензи-\nмидазол').replace('бензилпиперидин', 'бензил-\nпиперидин') for name in unique_inhibitors]
ax.set_xticklabels(labels, rotation=45, ha='right')

plt.ylabel('Метрика DTW', fontweight='bold')
plt.xlabel('')

plt.legend(title='Архитектура', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
filename = SAVE_DIR / 'all_models_comparison_dtw_presentation.png'
plt.savefig(filename, dpi=300, bbox_inches='tight')
plt.close(fig)